In [ ]:
!pip install duckdb geopandas shapely pyproj \
              xarray netCDF4 fsspec\
              folium branca rioxarray\
              matplotlib seaborn \
              requests folium

Import required libraries.

In [ ]:
import duckdb
import pandas as pd
import geopandas as gpd
import xarray as xr
import folium
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import folium
from folium.plugins import MarkerCluster
import fsspec

Let's check the DuckDB versio.

In [ ]:
duckdb.__version__

'1.3.2'

Creat DuckDB in-memory connection.

In [ ]:
con = duckdb.connect()

Install and load a few connections.

In [ ]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

Path to the parquet file which is saved on cloud.

In [ ]:
geonames_parquet = 'https://storage.googleapis.com/foss4g_workshop/geonames_asia_cities-del.parquet'

Try querying the DuckDB parquet file from the cloud.

In [ ]:
cities_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{geonames_parquet}')
    LIMIT 10
""").df()

cities_df


,geonameid,name,country_code,population,latitude,longitude
0,1163293,Tithwāl,IN,0,34.39351,73.77416
1,1163420,Thruti,IN,0,33.52682,74.15939
2,1163626,Thang,IN,1622,34.92740,76.79336
3,1167718,Pūnch,IN,28197,33.77033,74.09254
4,1171555,Mahmūd Khāneke,IN,0,30.66711,74.21704
5,1178984,Ganga Sāgar,IN,0,21.63933,88.07742
6,1181619,Chalunka,IN,0,34.82016,76.94212
7,1252646,Kilakarai,IN,38355,9.23183,78.78545
8,1252651,Zuvvaladinne,IN,0,14.80796,80.07045
9,1252653,Zunheboto,IN,29499,25.96667,94.51667


Let's see how many records we have in our parquet file.

In [ ]:
count_df = con.execute(f"""
    SELECT COUNT(*) AS total_records
    FROM read_parquet('{geonames_parquet}')
""").df()

count_df


,total_records
0,1839665


Here, we are just getting the summary. Getting number of cities and population in all countries.

In [ ]:
country_stats_df = con.execute(f"""
    SELECT
        country_code,
        COUNT(*) AS city_count,
        SUM(population) AS total_population
    FROM read_parquet('{geonames_parquet}')
    GROUP BY country_code
    ORDER BY total_population DESC
""").df()

country_stats_df


,country_code,city_count,total_population
0,CN,885611,705345423.0
1,IN,557948,392049916.0
2,JP,50799,146668257.0
3,ID,258053,82713003.0
4,TH,87254,23610267.0


Now, For each country, we sort cities by population and keep the top five. so, we are getting top 5 populated cities in each county.

In [ ]:
top_cities_df = con.execute(f"""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY country_code
                ORDER BY population DESC
            ) AS rank
        FROM read_parquet('{geonames_parquet}')
    )
    WHERE rank <= 5
""").df()

top_cities_df


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,geonameid,name,country_code,population,latitude,longitude,rank
0,1850147,Tokyo,JP,9733276,35.68950,139.69171,1
1,1848354,Yokohama,JP,3777491,35.43333,139.65000,2
2,1853909,Osaka,JP,2753862,34.69379,135.50107,3
3,1856057,Nagoya,JP,2332176,35.18147,136.90641,4
4,2128295,Sapporo,JP,1973832,43.06667,141.35000,5
5,1642911,Jakarta,ID,8540121,-6.21462,106.84513,1
6,1625822,Surabaya,ID,2874314,-7.24917,112.75083,2
7,1649378,Bekasi,ID,2564940,-6.23490,106.98960,3
8,1650357,Bandung,ID,2444160,-6.92222,107.60694,4
9,1214520,Medan,ID,2435252,3.58333,98.66667,5


Let's have some visualization. Start by creating a geodataframe using the lat/lon clolumns.

> Add blockquote



In [ ]:
gdf_top_cities = gpd.GeoDataFrame(
    top_cities_df,
    geometry=gpd.points_from_xy(
        top_cities_df.longitude,
        top_cities_df.latitude
    ),
    crs="EPSG:4326"  # WGS84 (lat/lon)
)

gdf_map = gdf_top_cities.copy()

Now, we are areating interactive map using matplotlib and flolium for visualizing these large cities.

In [ ]:
import matplotlib.pyplot as plt

countries = gdf_map["country_code"].unique()
cmap = plt.cm.get_cmap("tab20", len(countries))

color_map = {
    country: f"#{int(cmap(i)[0]*255):02x}{int(cmap(i)[1]*255):02x}{int(cmap(i)[2]*255):02x}"
    for i, country in enumerate(countries)
}
m = folium.Map(
    location=[30, 80],  # Asia-centered view
    zoom_start=4,
    tiles="CartoDB positron"
)
marker_cluster = MarkerCluster().add_to(m)

for _, row in gdf_map.iterrows():
    popup_text = f"""
    <b>City:</b> {row['name']}<br>
    <b>Country:</b> {row['country_code']}<br>
    <b>Population:</b> {int(row['population']):,}
    """

    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=6,
        color=color_map[row.country_code],
        fill=True,
        fill_color=color_map[row.country_code],
        fill_opacity=0.8,
        popup=popup_text
    ).add_to(marker_cluster)
m


/tmp/ipython-input-161674154.py:4: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab20", len(countries))


Taking a step ahead by bringing climate data from WorldClim to find climate based heat exposure of cities. WorldClim is a global climate dataset that provides long-term average climate conditions for the Earth.WorldClim shows typical climate conditions, not extreme daily events. WorldClim gives us global climate context, and GeoNames gives us human settlements, combining both lets us analyze climate exposure of cities. We are using averaged monthly maximum temperature data downloaded for 2020-2024 in GeoTiff formate. It was converted to COG using GDAL and uploaded on the cloud bucket.(~10km spatial resolution)

In [ ]:
import re
import rioxarray as rxr
BASE_URL = (
    "https://storage.googleapis.com/foss4g_workshop/"
    "wc2.1_cruts4.09_10m_tmax_2020-2024/cogs/"
)

YEARS = range(2020, 2025)
MONTHS = range(1, 13)
def extract_time_from_filename(url):
    match = re.search(r"_(\d{4}-\d{2})\.tif", url)
    if not match:
        raise ValueError(f"Could not parse date from filename: {url}")
    return pd.to_datetime(match.group(1))
files = [
    f"{BASE_URL}cog_wc2.1_cruts4.09_10m_tmax_{y}-{m:02d}.tif?alt=media"
    for y in YEARS
    for m in MONTHS
]
rasters = []
times = []
missing = []

for f in files:
    try:
        da = rxr.open_rasterio(f).squeeze()
        rasters.append(da)
        times.append(extract_time_from_filename(f))
    except Exception as e:
        missing.append(f)
da = xr.concat(rasters, dim="time")
da = da.assign_coords(time=times)
da = da.rename("tmax")
da = da.rename({
    "x": "longitude",
    "y": "latitude"})

print(da.dims)
print(da.sizes)



('time', 'latitude', 'longitude')
Frozen({'time': 60, 'latitude': 1080, 'longitude': 2160})


We are slicing the array spatially to keep it only for the asia bounding box.

In [ ]:
asia_bbox = {
    "lon_min": 25,
    "lon_max": 150,
    "lat_min": -10,
    "lat_max": 60}
da_asia = da.sel(
    longitude=slice(asia_bbox["lon_min"], asia_bbox["lon_max"]),
    latitude=slice(asia_bbox["lat_max"], asia_bbox["lat_min"]))



Creating a large table which can be further analyze and join with GeoNames using DuckDB. We are converting it to the pandas dataframe in different chunks. Chunking means processing a large dataset in smaller pieces instead of all at once.

In [ ]:
dfs = []
for t in da_asia.time:
    df = (
        da_asia.sel(time=t)
        .to_dataframe(name="tmax")
        .reset_index()
        .dropna()
    )
    dfs.append(df)
    print(dfs[0])

         latitude   longitude  band  spatial_ref       time       tmax
11      59.916667   26.916667     1            0 2020-01-01   8.000000
21      59.916667   28.583333     1            0 2020-01-01   4.000000
23      59.916667   28.916667     1            0 2020-01-01   7.000000
24      59.916667   29.083333     1            0 2020-01-01   3.428571
25      59.916667   29.250000     1            0 2020-01-01   3.000000
...           ...         ...   ...          ...        ...        ...
314995  -9.916667  149.250000     1            0 2020-01-01  26.875000
314996  -9.916667  149.416667     1            0 2020-01-01  25.937500
314997  -9.916667  149.583333     1            0 2020-01-01  26.687500
314998  -9.916667  149.750000     1            0 2020-01-01  30.142857
314999  -9.916667  149.916667     1            0 2020-01-01  31.000000

[184921 rows x 6 columns]
         latitude   longitude  band  spatial_ref       time       tmax
11      59.916667   26.916667     1            0 2

In [ ]:
print(type(dfs))
print(len(dfs))
print(type(dfs[0]))
print(dfs[0].shape)

<class 'list'>
60
<class 'pandas.core.frame.DataFrame'>
(184921, 6)


Now, Let's get the data in DuckDB table. We will creates a temporary view inside DuckDB. The 'register' function is a python utility of DuckDB, tells duckDB to treat the first chunk of pandas dataframe as DuckDB table and then creating a table using that chunk.

In [ ]:
con.register("chunk_df", dfs[0])
con.execute("CREATE TABLE climate_data AS SELECT * FROM chunk_df")

We process the climate data chunk by chunk to avoid loading everything into memory.Each chunk is temporarily registered as a table (chunk_df) inside DuckDB.
We then append that chunk into the main table (climate_data).This repeats until all chunks are stored safely in DuckDB.
Finally, we run a quick COUNT(*) query to confirm how many records were loaded.

In [ ]:
for df in dfs[1:]:
    con.register("chunk_df", df)
    con.execute("INSERT INTO climate_data SELECT * FROM chunk_df")
con.execute("SELECT COUNT(*) FROM climate_data").fetchone()[0]


11095260

Now, let's use this data to get temperature percentile for all cities within a country. In simple words - Within each country, which cities are experiencing relatively higher temperatures compared to other cities in the same country (during 2020 - 2024)?
So, Technically,

* We will assign each city a temperature value from the nearest climate grid cell.
* For each country, we rank cities relative to other cities in that country (We will compute temperature percentiles, not absolute thresholds)
* 90th percentile → city is hotter than 90% of cities in the same country
* Top 10% → relatively high heat exposure within national context

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE geonames_large_cities AS
SELECT *
FROM read_parquet('{geonames_parquet}')
WHERE population >= 1000000;
""")

WorldClim 10m data has a spatial resolution of 10 arc-minutes, which is approximately 0.1667 degrees.
Near the equator, this corresponds to a grid cell area of roughly 340 km². - [Documentation](https://www.worldclim.org/data/worldclim21.html)
Because the data is stored in geographic (latitude/longitude) coordinates, the actual cell area decreases toward higher latitudes.

Climate data is a regular spatial grid, while cities are point locations.To combine these two datasets efficiently, we will convert both into a shared grid index.

By dividing latitude and longitude by the known grid resolution (0.1667°) and flooring the result, we assign:

*   each climate record to a unique grid cell ID
*   each city to the grid cell it falls into

This turns a spatial containment problem into a simple integer join, which is fast, scalable, and avoids floating-point precision issues.

In [ ]:

GRID_SIZE = 0.1666667

con.execute(f"""
CREATE OR REPLACE TEMP TABLE cities_snapped AS
SELECT
  geonameid,
  name,
  country_code,
  latitude,
  longitude,
  population,
  CAST(FLOOR(latitude / 0.1666667) AS INTEGER) AS grid_y,
  CAST(FLOOR(longitude / 0.1666667) AS INTEGER) AS grid_x
FROM geonames_large_cities
""")

con.execute("""
SELECT *
FROM cities_snapped
LIMIT 10;
""").df()

,geonameid,name,country_code,latitude,longitude,population,grid_y,grid_x
0,1253102,Visakhapatnam,IN,17.68009,83.20161,1063178,106,499
1,1253133,Virār,IN,19.45591,72.81136,1222390,116,436
2,1253184,Vijayawada,IN,16.50745,80.64660,1143232,99,483
3,1253405,Varanasi,IN,25.31668,83.01041,1164404,151,498
4,1253573,Vadodara,IN,22.29941,73.20812,1822221,133,439
5,1254361,Tirunelveli,IN,8.72742,77.68380,1435844,52,466
6,1254388,Tiruchirappalli,IN,10.81550,78.69651,1022518,64,472
7,1254661,Thāne,IN,19.19704,72.96355,1841488,115,437
8,1254745,Teni,IN,10.01115,77.47772,1034724,60,464
9,1255364,Surat,IN,21.19594,72.83023,4591246,127,436


In [ ]:
GRID_SIZE = 0.1666667

con.execute(f"""
CREATE OR REPLACE TEMP TABLE climate_indexed AS
SELECT
  time,
  latitude,
  longitude,
  tmax,
  CAST(FLOOR(latitude / 0.1666667) AS INTEGER)  AS grid_y,
  CAST(FLOOR(longitude / 0.1666667) AS INTEGER) AS grid_x
  FROM climate_data
""")

con.execute("""
SELECT *
FROM climate_indexed
LIMIT 10;
""").df()

,time,latitude,longitude,tmax,grid_y,grid_x
0,2020-01-01,59.916667,26.916667,8.000000,359,161
1,2020-01-01,59.916667,28.583333,4.000000,359,171
2,2020-01-01,59.916667,28.916667,7.000000,359,173
3,2020-01-01,59.916667,29.083333,3.428571,359,174
4,2020-01-01,59.916667,29.250000,3.000000,359,175
5,2020-01-01,59.916667,29.416667,3.000000,359,176
6,2020-01-01,59.916667,29.583333,3.285714,359,177
7,2020-01-01,59.916667,29.750000,3.066667,359,178
8,2020-01-01,59.916667,29.916667,3.666667,359,179
9,2020-01-01,59.916667,30.083333,4.166667,359,180


Now, let's assign temperatures to the GeoNames cities.

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE city_climate AS
SELECT
  c.geonameid,
  c.name,
  c.country_code,
  c.population,
  c.latitude,
  c.longitude,
  cl.time,
  cl.tmax
FROM cities_snapped c
JOIN climate_indexed cl
  ON c.grid_y = cl.grid_y
 AND c.grid_x = cl.grid_x
""")
con.execute("""
SELECT *
FROM city_climate
LIMIT 10;
""").df()

,geonameid,name,country_code,population,latitude,longitude,time,tmax
0,1279259,Agra,IN,1430055,27.18333,78.01667,2020-01-01,20.8125
1,1816373,Bijie,CN,1137383,27.30193,105.28627,2020-01-01,7.8125
2,1269515,Jaipur,IN,3046163,26.91962,75.78781,2020-01-01,21.8750
3,1264733,Lucknow,IN,2472011,26.83928,80.92313,2020-01-01,21.6875
4,1808370,Hengyang,CN,1075516,26.88946,112.61888,2020-01-01,11.0000
5,8533133,Liupanshui,CN,1320825,26.59444,104.83333,2020-01-01,8.3125
6,1809461,Guiyang,CN,3037159,26.58333,106.71667,2020-01-01,9.8750
7,1267995,Kanpur,IN,2823249,26.46523,80.34975,2020-01-01,22.1250
8,1786217,Yongzhou,CN,1020715,26.42389,111.61306,2020-01-01,11.0000
9,1268865,Jodhpur,IN,1056191,26.26841,73.00594,2020-01-01,24.2500


Now, we are computing an average temperature for the city using all timestemps.So, we will have one value per city.

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE city_temp_summary AS
SELECT
  geonameid,
  name,
  country_code,
  ANY_VALUE(population) AS population,
  ANY_VALUE(latitude) AS latitude,
  ANY_VALUE(longitude) AS longitude,
  AVG(tmax) AS avg_tmax
FROM city_climate
GROUP BY geonameid, name, country_code;
""")

con.execute("""
SELECT *
FROM city_temp_summary
""").df()

,geonameid,name,country_code,population,latitude,longitude,avg_tmax
0,1269515,Jaipur,IN,3046163,26.91962,75.78781,32.544792
1,1264733,Lucknow,IN,2472011,26.83928,80.92313,32.361458
2,1808370,Hengyang,CN,1075516,26.88946,112.61888,23.639583
3,1786217,Yongzhou,CN,1020715,26.42389,111.61306,23.118750
4,1270583,Gwalior,IN,1054420,26.22983,78.17337,33.195833
...,...,...,...,...,...,...,...
243,1271951,Faridabad,IN,1414050,28.41124,77.31316,31.943750
244,1787858,Shangrao,CN,1116486,28.45179,117.94287,24.089583
245,1802875,Guankou,CN,1380000,28.15861,113.62709,22.322917
246,1783763,Zhuzhou,CN,1129687,27.83333,113.15000,22.797917


Compute percentile across cities within each country.

In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE city_with_percentile AS
SELECT
  *,
  PERCENT_RANK() OVER (
    PARTITION BY country_code
    ORDER BY avg_tmax
  ) AS temp_percentile
FROM city_temp_summary;
""")


Finally, we will get high heat cities per country.(top 10%)

In [ ]:
result_df = con.execute("""
SELECT
  geonameid,
  name,
  country_code,
  avg_tmax,
  temp_percentile
FROM city_with_percentile
WHERE temp_percentile >= 0.9
ORDER BY avg_tmax DESC;
""").df()
result_df

,geonameid,name,country_code,avg_tmax,temp_percentile
0,1253573,Vadodara,IN,34.746875,1.000000
1,1279233,Ahmedabad,IN,34.725000,0.982456
2,1268865,Jodhpur,IN,34.504167,0.964912
3,1254388,Tiruchirappalli,IN,34.283333,0.947368
4,12165956,Kallakurichi,IN,34.235417,0.929825
5,1264521,Madurai,IN,34.185417,0.912281
6,1625822,Surabaya,ID,33.255555,1.000000
7,1214520,Medan,ID,32.472917,0.933333
8,1796556,Sanya,CN,29.448810,1.000000
9,1809078,Haikou,CN,28.480952,0.993750


Now, let's create a point data of cities having population more than 1M with the short-tem average of maximum temperature as an attribute.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE city_temp_summary_spatial AS
SELECT
    geonameid,
    name,
    country_code,
    population,
    latitude,
    longitude,
    avg_tmax,
    ST_Point(longitude, latitude) AS geom
FROM city_temp_summary;
""")


In [ ]:
con.execute("""
SELECT
    ST_GeometryType(geom) AS geom_type,
    COUNT(*)
FROM city_temp_summary_spatial
GROUP BY geom_type;
""").fetchall()


[('POINT', 248)]

Export the data to geoparquet file and visualize in QGIS.


> Note : DuckDB does not yet support native GeoParquet writing.
When a table contains a GEOMETRY column and the target filename ends with .geo.parquet, DuckDB automatically routes the export through GDAL (via the spatial extension) to produce a valid GeoParquet file.

What it does internally,

```
COPY city_temp_summary_spatial
TO 'geonames_citiea.geo.parquet'
(
  FORMAT GDAL,
  DRIVER 'Parquet'
);

```

In [ ]:
con.execute("""
COPY city_temp_summary_spatial
TO 'geonames_citiea.geo.parquet'
""")


Now, we are getting the geoboundaries state/province data from the API. Reading it using geopandas and saving as geoparquet to be used for spatial aggregation and analysis with the cities file.
One can explore getting data directly from Geoparquete file from [overturemaps](https://docs.overturemaps.org/getting-data/).

In [ ]:
import requests
import geopandas as gpd
import pandas as pd

def get_country_boundaries(country_code, admin_level='ADM1'):
    api_url = f"https://www.geoboundaries.org/api/current/gbOpen/{country_code}/{admin_level}/"
    response = requests.get(api_url)

    if response.status_code == 200:
        data = response.json()
        gdf = gpd.read_file(data['gjDownloadURL'])
        gdf['country_code'] = country_code
        print(f"✓ {country_code}: {len(gdf)} boundaries")
        return gdf
    return None

# Get all 5 countries
print("Downloading from geoBoundaries API...\n")
countries = ['IND', 'CHN', 'JPN', 'IDN', 'THA']
all_states = [get_country_boundaries(c, 'ADM1') for c in countries]
all_states = [gdf for gdf in all_states if gdf is not None]

# Combine
asia_states_gdf = gpd.GeoDataFrame(pd.concat(all_states, ignore_index=True))

print(f"\n✓ Total: {len(asia_states_gdf)} boundaries")

# Clean up columns
asia_clean = asia_states_gdf[[
    'shapeName', 'shapeGroup', 'shapeType', 'shapeID',
    'country_code', 'geometry'
]].rename(columns={
    'shapeName': 'state_name',
    'shapeGroup': 'country_name',
    'shapeType': 'admin_level',
    'shapeID': 'state_id'
})

# Save directly as GeoParquet (GeoPandas handles this!)
print("\nSaving to GeoParquet...")
asia_clean.to_parquet('asia_states_geoboundaries.geoparquet')

import os
file_size = os.path.getsize('asia_states_geoboundaries.geoparquet') / 1024 / 1024
print(f"✓ Saved: asia_states_geoboundaries.geoparquet ({file_size:.2f} MB)")

# Summary
summary = asia_clean.groupby('country_code').size()
print(f"\nBreakdown by country:")
print(summary)

# Check disputed territories
india_states = asia_clean[asia_clean['country_code'] == 'IND']['state_name'].sort_values()
print(f"\nIndia states/UTs ({len(india_states)}):")
print(india_states.tolist())


✓ IND: 36 boundaries
✓ CHN: 34 boundaries
✓ JPN: 47 boundaries
✓ IDN: 34 boundaries
✓ THA: 77 boundaries

✓ Total: 228 boundaries

Saving to GeoParquet...
✓ Saved: asia_states_geoboundaries.geoparquet (58.81 MB)

Breakdown by country:
country_code
CHN    34
IDN    34
IND    36
JPN    47
THA    77
dtype: int64

India states/UTs (36):
['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunāchal Pradesh', 'Assam', 'Bihār', 'Chandīgarh', 'Chhattīsgarh', 'Delhi', 'Dādra and Nagar Haveli and Damān and Diu', 'Goa', 'Gujarāt', 'Haryāna', 'Himāchal Pradesh', 'Jammu and Kashmīr', 'Jhārkhand', 'Karnātaka', 'Kerala', 'Ladākh', 'Lakshadweep', 'Madhya Pradesh', 'Mahārāshtra', 'Manipur', 'Meghālaya', 'Mizoram', 'Nāgāland', 'Odisha', 'Puducherry', 'Punjab', 'Rājasthān', 'Sikkim', 'Tamil Nādu', 'Telangāna', 'Tripura', 'Uttar Pradesh', 'Uttarākhand', 'West Bengal']
